# Level 3: Alignment with Direct Preference Optimization (DPO)

After Supervised Fine-Tuning (SFT), a model is good at following instructions. But is it helpful, honest, and harmless? This is the goal of **alignment**.

The classic method for alignment is Reinforcement Learning from Human Feedback (RLHF). However, RLHF is complex, involving training a separate reward model and then using reinforcement learning (like PPO) to tune the LLM. It can be unstable and difficult to implement correctly.

**Direct Preference Optimization (DPO)** is a more recent, simpler, and more stable technique that achieves the same goal without needing a separate reward model.

### How DPO Works

DPO works by directly optimizing the language model based on human (or AI) preferences. It uses a dataset of preferences, where each example consists of:

- A `prompt`.
- A `chosen` response (the preferred one).
- A `rejected` response (the less-preferred one).

The model is trained to increase the likelihood of generating the `chosen` response while decreasing the likelihood of the `rejected` one. It's a simple classification loss on human preferences, but it's powerful enough to steer the model's behavior.

### Step 1: Install Dependencies

We'll need `trl` and `peft` again. Note that DPO can also be combined with QLoRA for memory efficiency.

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes trl datasets

### Step 2: Prepare the Preference Dataset

The dataset must have `prompt`, `chosen`, and `rejected` columns. We'll use a pre-formatted dataset from the Hub for this example.

In [ ]:
from datasets import load_dataset

# Load a preference dataset
# This dataset contains prompts and pairs of chosen/rejected responses
dataset = load_dataset("trl-internal-testing/hh-rlhf-helpful-base-trl-style", split="train[:100]")

# Let's look at an example
example = dataset[0]
print(f"Prompt: {example['prompt']}\n")
print(f"Chosen: {example['chosen']}\n")
print(f"Rejected: {example['rejected']}")

### Step 3: Load the SFT-Tuned Model

DPO training starts from a model that has already been instruction-tuned (SFT). We don't start from a base pre-trained model. We'll load a pre-trained SFT model and apply QLoRA to it for efficient tuning, just like in the previous notebook.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

model_id = "mistralai/Mistral-7B-Instruct-v0.2"

# QLoRA configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['q_proj', 'v_proj'] # Targeting attention projections
)

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

### Step 4: Train with DPOTrainer

The `DPOTrainer` from `trl` handles all the complexity. We just need to provide the model, tokenizer, and dataset.

In [ ]:
from trl import DPOTrainer
from transformers import TrainingArguments

# Training arguments
training_args = TrainingArguments(
    output_dir="./dpo-results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1, # Short training for demo
    logging_steps=10,
    fp16=True,
    optim="paged_adamw_8bit",
)

# DPO Trainer
dpo_trainer = DPOTrainer(
    model,
    model_ref=None, # No reference model needed for this setup
    args=training_args,
    beta=0.1, # The beta parameter is the strength of the preference loss
    train_dataset=dataset,
    tokenizer=tokenizer,
    peft_config=lora_config,
    max_prompt_length=512,
    max_length=1024,
)

# Start DPO training
dpo_trainer.train()

### Step 5: Inference

After DPO training, the model should be better aligned. Its responses should be more helpful and less likely to generate undesirable content.

In [ ]:
# The model used for training is the one to use for inference
prompt = "Hello, I have a question about my order."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = dpo_trainer.model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))